# Chatbot Lengkap

In [ ]:
pip install thefuzz

In [ ]:
import json
import re
import sys
from thefuzz import process, fuzz

class MasterRuleBasedChatbot:
    def __init__(self, default_year="2025"):
        BASE_PATH = "/content/drive/MyDrive/Skripsi/sources"

        # 1. Load semua data JSON
        self.skripsi_data = self.load_json(f"{BASE_PATH}/data_skripsi_advanced.json", [])
        self.kurikulum_data = self.load_json(f"{BASE_PATH}/kurikulum.json", {})

        # [UPDATE] Kalender Akademik gabungan 2 tahun ajaran: 2025/2026 & 2026/2027
        kalender_2526 = self.load_json(f"{BASE_PATH}/kalender_akademik.json", [])
        kalender_2627 = self.load_json(f"{BASE_PATH}/kalender_akademik_2627_intents.json", [])
        # Tandai asal tahun ajaran di tiap intent biar response-nya jelas
        for it in kalender_2526: it.setdefault('tahun_ajaran', '2025/2026')
        for it in kalender_2627: it.setdefault('tahun_ajaran', '2026/2027')
        self.kalender_data = kalender_2526 + kalender_2627

        self.dosen_data = self.load_json(f"{BASE_PATH}/dosen.json", [])
        self.informasi_umum_data = self.load_json(f"{BASE_PATH}/informasi_umum.json", [])

        # [BARU] Load data After Sidang (format: {"intents": [...]})
        after_sidang_map_raw = self.load_json(f"{BASE_PATH}/after_sidang_map_biru_merah_chatbot.json", {})
        after_sidang_sitei_raw = self.load_json(f"{BASE_PATH}/after_sidang_sitei_chatbot.json", {})
        self.after_sidang_data = (
            self._extract_intents(after_sidang_map_raw)
            + self._extract_intents(after_sidang_sitei_raw)
        )

        # [BARU] Load SOP Jurusan (KP) & SOP Skripsi (Bab 1-3)
        self.sop_jte_data = self.load_json(f"{BASE_PATH}/sop_jte_fixed.json", [])
        self.sop_skripsi_data = self.load_json(f"{BASE_PATH}/sop_skripsi.json", [])

        # 2. Aturan Rule-Based untuk Skripsi (Sebagai target Fuzzy Search)
        self.skripsi_rules = {
            "Kertas": ["kertas", "ukuran kertas", "hvs", "a4", "jenis kertas"],
            "Cetakan naskah": ["cetakan", "bolak-balik", "satu sisi", "single side", "muka"],
            "Jenis huruf": ["font", "jenis huruf", "times new roman", "tnr", "ukuran huruf"],
            "Jarak baris": ["spasi", "jarak baris", "jarak paragraf", "line spacing", "jarak judul"],
            "Batas tepi": ["margin", "batas tepi", "batas margin", "jarak tepi", "tepi atas", "tepi bawah", "tepi kiri", "tepi kanan", "kiri kanan"],
            "Sampul CD": ["sampul cd", "cover cd", "label cd"],
            "File CD": ["file cd", "nama file cd", "folder cd", "susunan file cd", "softcopy", "pdf"],
            "Halaman Sampul": ["sampul", "cover", "warna sampul", "karton", "halaman depan", "buffalo"],
            "Daftar Pustaka": ["daftar pustaka", "referensi", "apa style", "dafpus", "pustaka"],
            "Tabel": ["tabel", "format tabel", "judul tabel", "kolom tabel", "sumber tabel"],
            "Gambar": ["gambar", "ilustrasi", "grafik", "format gambar", "judul gambar"],
            "Abstrak": ["abstrak", "ringkasan", "kata kunci", "keyword"]
        }

        # 3. Variabel & Konfigurasi Kurikulum
        self.kurikulum_year = default_year
        self.sem_map_2018 = {
            "Keahlian I": "Semester 5", "Keahlian II": "Semester 6",
            "Keahlian III": "Semester 6", "Keahlian IV": "Semester 7",
            "Keahlian V": "Semester 7"
        }

        self.course_names = []
        self.course_details = {}
        self.group_names = []
        self.spec_map = {}

        if not self.load_curriculum(self.kurikulum_year):
            if self.kurikulum_data:
                first_key = list(self.kurikulum_data.keys())[0]
                self.load_curriculum(first_key)

    def load_json(self, filepath, default_value):
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                return json.load(f)
        except FileNotFoundError:
            print(f"⚠️ Peringatan: File {filepath} tidak ditemukan.")
            return default_value

    def _extract_intents(self, raw):
        """Ambil list intents dari JSON yang mungkin dibungkus {'intents': [...]} atau langsung list."""
        if isinstance(raw, dict):
            return raw.get('intents', [])
        if isinstance(raw, list):
            return raw
        return []

    # ==========================================
    # LOGIKA KURIKULUM (Indexing & Processing)
    # ==========================================
    def load_curriculum(self, year):
        if year not in self.kurikulum_data: return False

        self.kurikulum_year = year
        data_active = self.kurikulum_data[year]

        self.course_names = []
        self.course_details = {}
        self.group_names = []
        self.spec_map = {}

        for group in data_active:
            g_name = group.get('group_name', 'Unknown')
            self.group_names.append(g_name)
            is_semester = "SEMESTER" in g_name.upper()

            for course in group.get('courses', []):
                c_name = course.get('name', '')
                c_no = course.get('no', '')

                real_semester = g_name if is_semester else self.sem_map_2018.get(str(c_no), "Semester Tidak Diketahui")

                if c_name:
                    self.course_names.append(c_name)
                    self.course_details[c_name] = {
                        "sks": course.get('sks', '0'), "group": g_name,
                        "semester_info": real_semester, "code": course.get('code', '-'), "no_label": c_no
                    }

                if self.kurikulum_year == "2018" and not is_semester and "Keahlian" in str(c_no):
                    if c_no not in self.spec_map: self.spec_map[c_no] = []
                    self.spec_map[c_no].append({
                        "course_name": c_name, "concentration": g_name, "semester": self.sem_map_2018.get(c_no, "")
                    })
        return True

    def get_semester_info(self, semester_num):
        target = f"SEMESTER {semester_num}"
        for group in self.kurikulum_data.get(self.kurikulum_year, []):
            if group['group_name'].upper() == target:
                total_sks = group.get('total_sks', '0')
                response = f"📚 **{target} (Kurikulum {self.kurikulum_year})**\nTotal Beban: {total_sks} SKS\n\n"
                for c in group['courses']:
                    if self.kurikulum_year == "2018" and c['name'] in self.spec_map:
                        response += f"🔻 **{c['name']} ({c['sks']} SKS) -> Pilih Konsentrasi:**\n"
                        for opt in self.spec_map[c['name']]: response += f"   - {opt['course_name']} ({opt['concentration']})\n"
                    else:
                        response += f"- [{c.get('code', '-')}] {c['name']} ({c['sks']} SKS)\n"
                return response
        return f"Maaf, data semester {semester_num} tidak ditemukan di Kurikulum {self.kurikulum_year}."

    # ==========================================
    # HELPER FUZZY SEARCH MULTI-DOMAIN
    # ==========================================
    def fuzzy_search_intent(self, user_query, data_list, threshold=80):
        """Mencari intent pada data JSON yang punya list 'keywords'."""
        best_item = None
        highest_score = 0
        for item in data_list:
            keywords = item.get('keywords', []) or item.get('keyword', [])
            if not keywords: continue

            match = process.extractOne(user_query, keywords, scorer=fuzz.token_set_ratio)
            if match and match[1] > highest_score:
                highest_score = match[1]
                best_item = item

        if highest_score >= threshold:
            return best_item
        return None

    def fuzzy_search_skripsi_category(self, user_query, threshold=80):
        """Mencari kategori Skripsi dari rules dictionary"""
        best_category = None
        highest_score = 0
        for category, keywords in self.skripsi_rules.items():
            match = process.extractOne(user_query, keywords, scorer=fuzz.token_set_ratio)
            if match and match[1] > highest_score:
                highest_score = match[1]
                best_category = category

        if highest_score >= threshold:
            return best_category
        return None

    def fuzzy_search_sop(self, user_query, data_list, threshold=70, limit=2):
        """Mencari chunk SOP (sop_jte/sop_skripsi) berdasarkan topik_utama/sub_topik/konten."""
        scored = []
        for item in data_list:
            targets = [
                str(item.get('topik_utama', '')),
                str(item.get('sub_topik', '')),
                str(item.get('full_context', '')),
            ]
            konten = str(item.get('konten', ''))
            if konten:
                # Potong konten biar fuzzy tetap cepat & relevan
                targets.append(konten[:400])

            targets = [t for t in targets if t]
            if not targets: continue

            match = process.extractOne(user_query, targets, scorer=fuzz.token_set_ratio)
            if match and match[1] >= threshold:
                scored.append((match[1], item))

        if not scored: return []
        scored.sort(key=lambda x: x[0], reverse=True)
        return [item for _, item in scored[:limit]]

    def fuzzy_search_curriculum(self, user_query):
        """Mencari nama Matkul atau Peminatan"""
        if not self.course_names: return None

        best_group, group_score = process.extractOne(user_query, self.group_names, scorer=fuzz.token_set_ratio)
        if group_score > 75 and "SEMESTER" not in best_group:
            for group in self.kurikulum_data.get(self.kurikulum_year, []):
                if group['group_name'] == best_group:
                    resp = f"📂 **Peminatan: {best_group} ({self.kurikulum_year})**\n\n"
                    for c in group['courses']: resp += f"- {c['name']} ({c['sks']} SKS)\n"
                    return resp

        matches = process.extract(user_query, self.course_names, limit=3, scorer=fuzz.token_set_ratio)
        valid_matches = [m for m in matches if m[1] > 65]

        if not valid_matches: return None

        response = f"🔍 **Hasil Pencarian Mata Kuliah ({self.kurikulum_year}):**\n"
        for name, score in valid_matches:
            details = self.course_details[name]
            response += f"\n📖 **{name}**\n   • Kode : {details['code']}\n   • SKS  : {details['sks']}\n"
            if "Keahlian" in str(details['no_label']) and self.kurikulum_year == "2018":
                response += f"   • Kategori : {details['no_label']} ({details['semester_info']})\n   • Grup     : {details['group']}\n"
            else:
                response += f"   • Semester : {details['group']}\n"
        return response

    # ==========================================
    # ROUTER UTAMA
    # ==========================================
    def get_response(self, user_input):
        cleaned_input = user_input.lower().strip()

        # 1. Perintah Eksplisit / Sistem (Tidak pakai Fuzzy agar tidak nyasar)
        if "ganti" in cleaned_input:
            if "2018" in cleaned_input:
                return "✅ Berhasil beralih ke Kurikulum 2018." if self.load_curriculum("2018") else "❌ Data Kurikulum 2018 tidak ada."
            elif "2025" in cleaned_input:
                return "✅ Berhasil beralih ke Kurikulum 2025." if self.load_curriculum("2025") else "❌ Data Kurikulum 2025 tidak ada."

        if "total" in cleaned_input and "sks" in cleaned_input:
            return f"🤖 Total SKS yang harus ditempuh berdasarkan Kurikulum {self.kurikulum_year} adalah 144 SKS."

        sem_match = re.search(r'(?:sem|semester)\s*(\d+)', cleaned_input)
        if sem_match:
            return self.get_semester_info(sem_match.group(1))

        # 2. Cek Data Informasi Umum
        best_info = self.fuzzy_search_intent(cleaned_input, self.informasi_umum_data, threshold=80)
        if best_info:
            return f"ℹ️ **Informasi:**\n{best_info['response']}"

        # 3. Cek Data Dosen
        best_dosen = self.fuzzy_search_intent(cleaned_input, self.dosen_data, threshold=80)
        if best_dosen:
            return f"👨‍🏫 **Informasi Dosen:**\n{best_dosen['response']}"

        # 4. Cek Data Kalender Akademik (gabungan 2025/2026 & 2026/2027)
        best_kalender = self.fuzzy_search_intent(cleaned_input, self.kalender_data, threshold=80)
        if best_kalender:
            ta = best_kalender.get('tahun_ajaran', '')
            header = f"📅 **Informasi Akademik (TA {ta}):**" if ta else "📅 **Informasi Akademik:**"
            return f"{header}\n{best_kalender['response']}"

        # 5. [BARU] Cek Data After Sidang (Map Biru/Merah & SITEI)
        best_after = self.fuzzy_search_intent(cleaned_input, self.after_sidang_data, threshold=80)
        if best_after:
            return f"📂 **After Sidang:**\n{best_after['response']}"

        # 6. Cek Data Pedoman Penulisan Skripsi (kategori rules)
        matched_skripsi_cat = self.fuzzy_search_skripsi_category(cleaned_input, threshold=80)
        if matched_skripsi_cat:
            responses = []
            for chunk in self.skripsi_data:
                if matched_skripsi_cat.lower() in chunk.get('sub_topik', '').lower() or matched_skripsi_cat.lower() in chunk.get('topik_utama', '').lower():
                    responses.append(f"📌 **{chunk.get('full_context', 'Info')}**\n{chunk.get('konten', '')}")
            if responses: return "\n\n".join(responses)

        # 7. [BARU] Cek SOP Skripsi (Bab 1-3) & SOP JTE (Kerja Praktek)
        sop_results = []
        sop_skripsi_hits = self.fuzzy_search_sop(cleaned_input, self.sop_skripsi_data, threshold=70, limit=2)
        sop_jte_hits = self.fuzzy_search_sop(cleaned_input, self.sop_jte_data, threshold=70, limit=2)

        for chunk in sop_skripsi_hits:
            sop_results.append(f"📘 **SOP Skripsi — {chunk.get('full_context', 'Info')}**\n{chunk.get('konten', '')}")
        for chunk in sop_jte_hits:
            sop_results.append(f"📗 **SOP JTE — {chunk.get('full_context', 'Info')}**\n{chunk.get('konten', '')}")

        if sop_results:
            return "\n\n".join(sop_results)

        # 8. Cek Data Kurikulum Matkul (Fuzzy Search dengan threshold 65)
        fuzzy_kurikulum = self.fuzzy_search_curriculum(cleaned_input)
        if fuzzy_kurikulum:
            return fuzzy_kurikulum

        # 9. Fallback
        return ("Maaf, saya tidak menemukan jawaban yang relevan. Coba periksa ejaanmu atau gunakan perintah:\n"
                "- 'semester 5' atau 'matkul algoritma' (Kurikulum)\n"
                "- 'kapan wisuda 128' atau 'jadwal uts' (Kalender Akademik)\n"
                "- 'dosen pak irsan' atau 'kaprodi ti' (Dosen)\n"
                "- 'aturan margin skripsi' (Pedoman Skripsi)\n"
                "- 'map biru' atau 'alur bebas lab' (After Sidang)\n"
                "- 'prosedur seminar proposal' atau 'pendaftaran KP' (SOP)\n"
                "- 'alur pendaftaran', 'beasiswa' atau 'biaya ukt' (Informasi Umum)\n"
                f"- 'ganti 2018' (Ubah versi kurikulum, saat ini: {self.kurikulum_year})")

# --- BLOK EKSEKUSI ---
if __name__ == "__main__":
    bot = MasterRuleBasedChatbot(default_year="2025")

    print("="*60)
    print("🤖 Sistem Chatbot Multi-Data Full Fuzzy Search Aktif!")
    print(f"Status: Kurikulum Aktif {bot.kurikulum_year}")
    print(f"Sumber Data: Kurikulum, Kalender 25/26 + 26/27, Dosen, Info Umum,")
    print(f"             Pedoman Skripsi, After Sidang, SOP JTE, SOP Skripsi")
    print("="*60)

    while True:
        try:
            user_input = input("\nKamu: ").strip()
            if not user_input: continue
            if user_input.lower() in ['keluar', 'exit', 'quit']:
                print("Sampai jumpa! 👋")
                break

            response = bot.get_response(user_input)
            print(f"Bot:\n{response}")

        except KeyboardInterrupt:
            print("\nKeluar paksa. Sampai jumpa!")
            break